<a href="https://colab.research.google.com/github/YF4002/PDF-Retrieval-Augmented-Generation-RAG-Pipeline/blob/main/Final_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Dependencies

!pip install -q gradio pypdf PyMuPDF pytesseract pillow faiss-cpu sentence-transformers transformers accelerate

# Imports

import gradio as gr
import fitz  # PyMuPDF for PDF handling
import pytesseract  # OCR for scanned PDFs
from PIL import Image
from io import BytesIO
import faiss  # Vector similarity search
import numpy as np
from sentence_transformers import SentenceTransformer  # Embeddings
from transformers import pipeline  # RAG model
from typing import List, Dict, Tuple

# Load Models
# Embedding model for vectorizing text chunks
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# RAG model for generating answers from retrieved chunks
rag_model = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",  # Lightweight and fast for Colab
    device_map="auto",
    max_new_tokens=300
)

# Helper Class for Chunk Metadata
class ChunkMetadata:
    """Stores info about each text chunk."""
    def __init__(self, text: str, page_start: int, page_end: int, doc_type: str, source_id: str):
        self.text = text
        self.page_start = page_start
        self.page_end = page_end
        self.doc_type = doc_type
        self.source_id = source_id

# Document Processing Functions
def extract_text_from_pdf(pdf_bytes: bytes) -> List[Tuple[str, int, str]]:
    """
    Extracts text from each PDF page.
    Falls back to OCR if no text is found.
    Returns list of tuples: (text, page_number, source_id)
    """
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    results = []
    for i, page in enumerate(doc):
        text = page.get_text("text")
        if not text.strip():  # fallback to OCR if page has no text
            pix = page.get_pixmap()
            img = Image.open(BytesIO(pix.tobytes("png")))
            text = pytesseract.image_to_string(img)
        results.append((text, i + 1, f"page_{i+1}"))
    return results

def chunk_text(text: str, chunk_size=300, overlap=50) -> List[str]:
    """
    Split text into overlapping chunks for embeddings.
    """
    words, chunks, start = text.split(), [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

def classify_doc_type(text: str) -> str:
    """
    Simple classifier to detect document type.
    """
    lower = text.lower()
    if "invoice" in lower: return "Invoice"
    if "report" in lower: return "Report"
    if "contract" in lower: return "Contract"
    return "General"

# FAISS Indexing
index = None  # FAISS index object
all_chunks: List[ChunkMetadata] = []  # Stores all chunks for retrieval

def build_faiss_index(chunks: List[ChunkMetadata]):
    """
    Build a FAISS index from text chunk embeddings.
    """
    global index, all_chunks
    all_chunks = chunks
    embeddings = embed_model.encode([c.text for c in chunks])
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings).astype("float32"))

def retrieve(query: str, k=3) -> List[Tuple[ChunkMetadata, float]]:
    """
    Retrieve top-k most relevant chunks from FAISS index.
    Returns list of (ChunkMetadata, relevance_score).
    """
    query_emb = embed_model.encode([query]).astype("float32")
    D, I = index.search(query_emb, k)
    return [(all_chunks[idx], float(1/(1+d))) for idx, d in zip(I[0], D[0]) if idx != -1]

# Answer Generation
def generate_answer_with_sources(query: str, retrieved_chunks: List[Tuple[ChunkMetadata, float]]) -> Dict:
    """
    Generate answer using RAG model based on retrieved chunks.
    Includes sources and confidence score.
    """
    if not retrieved_chunks:
        return {
            "answer": "I couldn't find relevant information.",
            "sources": [],
            "confidence": 0.0
        }

    context_parts, sources = [], []
    for chunk_meta, score in retrieved_chunks:
        # Include doc type and page info in context
        context_parts.append(f"[From {chunk_meta.doc_type}, Page {chunk_meta.page_start}]\n{chunk_meta.text}\n")
        sources.append({
            "doc_type": chunk_meta.doc_type,
            "pages": f"{chunk_meta.page_start}-{chunk_meta.page_end}",
            "relevance": f"{score:.2%}",
            "preview": chunk_meta.text[:120] + "..."
        })

    context = "\n".join(context_parts)
    prompt = f"""
Use the context to answer the question.

Context:
{context}

Question: {query}

Instructions:
1. Base your answer ONLY on the context.
2. Cite document type + page numbers.
3. If not enough info, say so.
"""
    response = rag_model(prompt)[0]["generated_text"]
    avg_score = sum(s for _, s in retrieved_chunks) / len(retrieved_chunks)

    return {
        "answer": response.strip(),
        "sources": sources,
        "confidence": avg_score,
        "chunks_used": len(retrieved_chunks)
    }

# Gradio UI Functions
def process_pdf(pdf_bytes):
    """
    Extract text, chunk, classify, and build FAISS index.
    """
    pages = extract_text_from_pdf(pdf_bytes)
    chunks = []
    for text, page_num, source_id in pages:
        doc_type = classify_doc_type(text)
        for chunk in chunk_text(text):
            chunks.append(ChunkMetadata(
                text=chunk,
                page_start=page_num,
                page_end=page_num,
                doc_type=doc_type,
                source_id=source_id
            ))
    build_faiss_index(chunks)
    return f"✅ Indexed {len(chunks)} chunks from {len(pages)} pages."

def chat(query, history):
    """
    Handles chatbot conversation: retrieve + generate answer + append to chat history.
    """
    retrieved = retrieve(query, k=3)
    result = generate_answer_with_sources(query, retrieved)
    sources_str = "\n".join([f"- {s['doc_type']} (pages {s['pages']}, relevance {s['relevance']})" for s in result["sources"]])
    answer = f"{result['answer']}\n\n📚 Sources:\n{sources_str}\n\nConfidence: {result['confidence']:.2%}"
    history = history + [(query, answer)]
    return history, history

# Gradio UI Layout
with gr.Blocks() as demo:
    gr.Markdown("## 📑 RAG Chatbot — Demo")

    with gr.Tab("Upload Document"):
        pdf_input = gr.File(type="binary", file_types=[".pdf"], label="Upload PDF")
        process_btn = gr.Button("Process PDF")
        status_output = gr.Textbox(label="Status")
        process_btn.click(process_pdf, inputs=pdf_input, outputs=status_output)

    with gr.Tab("Chat"):
        chatbot = gr.Chatbot()
        msg = gr.Textbox(label="Ask a question about your documents")
        clear = gr.Button("Clear Chat")
        state = gr.State([])
        msg.submit(chat, inputs=[msg, state], outputs=[chatbot, state])
        clear.click(lambda: ([], []), None, [chatbot, state])

# Launch UI with public shareable link
demo.launch(share=True)
